# Model 2: Flan-T5-large, generating the type and the spoiler

**The question:** can a sequence-to-sequence model that *writes* the spoiler do better than one that points at a span, and can it predict the type at the same time?

**`google/flan-t5-large`** (783M parameters, an instruction-tuned T5) reads the post and the article and writes the spoiler type followed by the spoiler, as one line:

| Gold | Target the model learns to write |
|---|---|
| phrase: *Gwyneth Paltrow* | `phrase: Gwyneth Paltrow` |
| multi: *bacon*, *eggs*, *toast* | `multi: bacon \| eggs \| toast` |

So one model does both subtasks, and a multi spoiler is any number of items separated by `|` (one T5 token that never occurs in a gold spoiler). Unlike the QA models, it isn't restricted to copying a span, and it doesn't need the separate type classifier.

**Input:** `question: <post> context: <title + paragraphs>`, cut to 1,024 tokens (T5's relative position encoding allows inputs longer than its 512-token pre-training). The cut hides every spoiler span of about 3% of training posts; those posts are left out of training, because their targets can't be learned from the input.

**Learning-rate sweep:** 5e-5, 1e-4, 3e-4. Batch 4 with 4 accumulation steps (effective 16), 5 epochs, AdamW with weight decay 0.01, linear schedule with 10% warmup, bf16, gradient checkpointing (to fit 1,024-token inputs in memory). Targets are at most 192 tokens (covers about 98% of multi spoilers). The best epoch is chosen by dev BLEU of the generated spoilers, with beam search (4 beams).

**Running it on the L4 VM** (about 9 hours: each learning rate takes about 2.8 hours):
```bash
cd ~/project/milestone3
source ~/project/.venv/bin/activate
pip install -r requirements.txt               # once for all Milestone 3 notebooks
cp -r ../milestone2/tools .                   # the official METEOR jar, already downloaded in Milestone 2
tmux new -s m3                                # detach with Ctrl-b d, reattach with: tmux attach -t m3
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 2-flan-t5.ipynb
```
`bash run-all.sh` runs all Milestone 3 notebooks one after the other instead. A learning rate that has finished (its `runs/.../run_info.json` exists) is skipped when the notebook is run again, so an interrupted sweep picks up where it stopped.

**Smoke test** (a few minutes on a laptop CPU; on the VM it uses the GPU): `SMOKE=1 jupyter nbconvert --to notebook --execute 2-flan-t5.ipynb --output smoke-run.ipynb`. It swaps in a tiny model, trains 2 steps on 16 posts and writes everything under `smoke/`.

**Outputs:** checkpoints in `runs/flan-t5-lr*/` (not committed); `output/flan-t5-sweep.json`, predictions `output/flan-t5-{dev,test}.jsonl` and `output/flan-t5-goldtypes-test.jsonl`, and scores in `output/scores/`.

## Setup

Imports and run settings. `SMOKE=1` switches to the tiny test run with `google/flan-t5-small`, shorter inputs and 2 beams.

In [1]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, TrainingArguments,
                          set_seed)
from transformers.utils import logging as hf_logging

from spoilerqa.data import DEFAULT_DATA_DIR, TYPES, load_jsonl, load_split, post_text, spoiler_type, write_jsonl
from spoilerqa.metrics import as_text, bleu, ensure_nltk_data

hf_logging.set_verbosity_error()
ensure_nltk_data()
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE", "0") == "1"
DATA_DIR = Path(os.environ.get("DATA_DIR", DEFAULT_DATA_DIR))
WORK_DIR = Path("smoke") if SMOKE else Path(".")
RUNS_DIR, OUTPUT_DIR = WORK_DIR / "runs", WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
M2_DIR = Path("inputs/milestone2")  # frozen copies of the Milestone 2 outputs we reuse
SEED = 42

HAS_GPU = torch.cuda.is_available()
USE_BF16 = HAS_GPU and torch.cuda.is_bf16_supported()
DEVICE = "cuda" if HAS_GPU else "cpu"  # a laptop runs the smoke test on the CPU

# Smoke-test limits (None / -1 = use everything).
MAX_TRAIN = 16 if SMOKE else None
MAX_EVAL = 8 if SMOKE else None
MAX_STEPS = 2 if SMOKE else -1

from spoilerqa.gen import (GenerationEvalTrainer, parse_t5, seq2seq_generate, spoiler_visible, t5_input, t5_target,
                           to_prediction)

NAME = "flan-t5"
MODEL = "google/flan-t5-small" if SMOKE else "google/flan-t5-large"
LEARNING_RATES = [5e-5, 1e-4, 3e-4]
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 5, 4, (1 if SMOKE else 4)
MAX_SOURCE = 256 if SMOKE else 1024   # input tokens
MAX_TARGET = 32 if SMOKE else 192     # output tokens
NUM_BEAMS = 2 if SMOKE else 4
GEN_BATCH = 16                        # posts per generation batch

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| bf16:", USE_BF16, "| smoke test:", SMOKE,
      "| model:", MODEL)

/home/dhanush/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA L4 | bf16: True | smoke test: False | model: google/flan-t5-large


/home/dhanush/project/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/dhanush/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Load the splits. In the smoke test we keep only the first few posts of each.

In [2]:
train = load_split(DATA_DIR, "train")[:MAX_TRAIN]
dev = load_split(DATA_DIR, "dev")[:MAX_EVAL]
test = load_split(DATA_DIR, "test")[:MAX_EVAL]
print({"train": len(train), "dev": len(dev), "test": len(test)})

{'train': 3157, 'dev': 794, 'test': 1000}


Build the training examples: the source text (with the article cut to fit 1,024 tokens) and the target text. Posts whose every spoiler span falls after the cut are left out of training. The cell prints how many, plus a few example targets.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)


def t5_features(records):
    features, hidden = [], 0
    for r in records:
        source, kept = t5_input(tokenizer, r, MAX_SOURCE)
        if not spoiler_visible(r, kept):
            hidden += 1
            continue
        enc = tokenizer(source, max_length=MAX_SOURCE, truncation=True)
        enc["labels"] = tokenizer(text_target=t5_target(r), max_length=MAX_TARGET, truncation=True)["input_ids"]
        features.append(dict(enc))
    return features, hidden


train_features, hidden = t5_features(train)
dev_features, _ = t5_features(dev)
print(f"train: {len(train)} posts -> {len(train_features)} examples ({hidden} left out: spoiler after the cut)")
print("input tokens (median / max):", int(pd.Series([len(f["input_ids"]) for f in train_features]).median()),
      "/", max(len(f["input_ids"]) for f in train_features))
for t in TYPES:
    r = next((x for x in train if spoiler_type(x) == t), None)
    if r:
        print(f"  {t:8} -> {t5_target(r)[:110]}")

train: 3157 posts -> 3072 examples (85 left out: spoiler after the cut)
input tokens (median / max): 526 / 1024
  phrase   -> phrase: 2070
  passage  -> passage: how about that morning we go throw?
  multi    -> multi: Purpose connects us to something bigger and in doing so makes us right sized | be ruthless with your "N


**Prediction function.** It generates with beam search, longest inputs first, and turns each output back into a type and a spoiler (`parse_t5`). If the output doesn't start with a valid type, the type is guessed from the spoiler: several items → multi, at most 5 words → phrase, else passage.

With `use_gold_types=True` the output is forced to start with the gold type (e.g. `passage:`) and the model writes only the spoiler. That is the analysis counterpart of the QA models' gold-type decoding: it shows how much wrong types cost.

In [4]:
def t5_predict(model, records, use_gold_types=False):
    sources = [t5_input(tokenizer, r, MAX_SOURCE)[0] for r in records]
    if use_gold_types:
        groups = {t: [i for i, r in enumerate(records) if spoiler_type(r) == t] for t in TYPES}
    else:
        groups = {None: list(range(len(records)))}
    texts = [None] * len(records)
    for t, idx in groups.items():
        if not idx:
            continue
        outputs = seq2seq_generate(model, tokenizer, [sources[i] for i in idx], MAX_SOURCE, MAX_TARGET, NUM_BEAMS,
                                   GEN_BATCH, decoder_prefix=f"{t}:" if t else None, bf16=USE_BF16)
        for i, text in zip(idx, outputs):
            texts[i] = text
    return [to_prediction(r["uuid"], *parse_t5(text)) for r, text in zip(records, texts)]

One training run for one learning rate. The trainer's evaluation (`GenerationEvalTrainer` in `spoilerqa/gen.py`) generates all dev spoilers after every epoch and reports BLEU and type balanced accuracy; the epoch with the best dev BLEU is kept. That dev BLEU is end to end, since the model writes the type itself. Similar-length inputs are batched together to save padding. The best model goes to `runs/flan-t5-lr<lr>/best`, the per-epoch scores to `run_info.json`; a run whose `run_info.json` exists is skipped.

In [5]:
def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()



def epoch_or_steps():
    # Evaluate and save after every epoch; the smoke test does it after its 2 steps instead.
    if MAX_STEPS < 0:
        return dict(eval_strategy="epoch", save_strategy="epoch")
    return dict(eval_strategy="steps", eval_steps=MAX_STEPS, save_strategy="steps", save_steps=MAX_STEPS)


def train_t5(learning_rate):
    run_dir = RUNS_DIR / f"{NAME}-lr{learning_rate:g}"
    if (run_dir / "run_info.json").exists():
        print("already trained, reusing", run_dir / "run_info.json")
        return json.loads((run_dir / "run_info.json").read_text())
    set_seed(SEED)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
    args = TrainingArguments(
        output_dir=str(run_dir), learning_rate=learning_rate, num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        weight_decay=0.01, warmup_steps=0.1, lr_scheduler_type="linear",  # warmup_steps < 1 means a ratio
        gradient_checkpointing=True, train_sampling_strategy="group_by_length",
        **epoch_or_steps(), save_total_limit=2, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="bleu", greater_is_better=True,
        logging_steps=25, bf16=USE_BF16, use_cpu=not HAS_GPU, report_to="none", seed=SEED,
    )
    trainer = GenerationEvalTrainer(
        model=model, args=args, train_dataset=train_features, eval_dataset=dev_features,
        processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        predict_fn=t5_predict, dev_records=dev,
    )
    trainer.train()
    trainer.save_model(str(run_dir / "best"))
    history = trainer.state.log_history
    per_epoch = [h["eval_bleu"] for h in history if "eval_bleu" in h]
    info = {"learning_rate": learning_rate, "dev_bleu": max(per_epoch), "dev_bleu_per_epoch": per_epoch,
            "dev_type_balanced_accuracy_per_epoch": [h["eval_type_balanced_accuracy"] for h in history
                                                     if "eval_type_balanced_accuracy" in h],
            "train_minutes": round(next(h["train_runtime"] for h in history if "train_runtime" in h) / 60, 1),
            "run_dir": str(run_dir)}
    (run_dir / "run_info.json").write_text(json.dumps(info, indent=2) + "\n")
    del trainer, model
    free_gpu()
    return info

**The learning-rate sweep.** Dev BLEU and type balanced accuracy after every epoch, saved to `output/flan-t5-sweep.json`.

In [6]:
sweep = []
for lr in LEARNING_RATES:
    print(f"=== learning rate {lr:g} ===")
    sweep.append(train_t5(lr))

(OUTPUT_DIR / f"{NAME}-sweep.json").write_text(json.dumps(sweep, indent=2) + "\n")
pd.DataFrame(sweep)[["learning_rate", "dev_bleu", "dev_bleu_per_epoch", "dev_type_balanced_accuracy_per_epoch",
                     "train_minutes"]]

=== learning rate 5e-05 ===
already trained, reusing runs/flan-t5-lr5e-05/run_info.json
=== learning rate 0.0001 ===


Loading weights: 100%|██████████| 558/558 [00:01<00:00, 343.76it/s]


{'loss': '5.682', 'grad_norm': '16.89', 'learning_rate': '2.5e-05', 'epoch': '0.1302'}
{'loss': '3.175', 'grad_norm': '9.06', 'learning_rate': '5.104e-05', 'epoch': '0.2604'}
{'loss': '2.38', 'grad_norm': '12.49', 'learning_rate': '7.708e-05', 'epoch': '0.3906'}
{'loss': '1.802', 'grad_norm': '5.597', 'learning_rate': '9.965e-05', 'epoch': '0.5208'}
{'loss': '1.682', 'grad_norm': '8.403', 'learning_rate': '9.676e-05', 'epoch': '0.651'}
{'loss': '1.574', 'grad_norm': '5.495', 'learning_rate': '9.387e-05', 'epoch': '0.7812'}
{'loss': '1.445', 'grad_norm': '4.681', 'learning_rate': '9.097e-05', 'epoch': '0.9115'}


/home/dhanush/project/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


{'eval_bleu': '0.2741', 'eval_type_balanced_accuracy': '0.4572', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.71s/it]


... (21 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.75s/it]


{'train_runtime': '1.005e+04', 'train_samples_per_second': '1.528', 'train_steps_per_second': '0.095', 'train_loss': '0.8851', 'epoch': '5'}


Writing model shards: 100%|██████████| 1/1 [00:11<00:00, 11.81s/it]


,learning_rate,dev_bleu,dev_bleu_per_epoch,dev_type_balanced_accuracy_per_epoch,train_minutes
0,0.00005,0.415681,"[0.262354, 0.32887, 0.344991, 0.415681, 0.410808]","[0.421861, 0.534, 0.569075, 0.648221, 0.65254]",170.6
1,0.00010,0.426227,"[0.2741, 0.358148, 0.408435, 0.423792, 0.426227]","[0.457197, 0.652822, 0.689615, 0.710997, 0.720697]",165.9
2,0.00030,0.450494,"[0.332521, 0.362007, 0.42525, 0.443471, 0.450494]","[0.569855, 0.708687, 0.729942, 0.739543, 0.764806]",167.6


## Prediction

Load the best run and generate dev and test, plus test with the gold types forced (analysis only). Files: `flan-t5-{dev,test}.jsonl` and `flan-t5-goldtypes-test.jsonl`.

In [7]:
best_run = max(sweep, key=lambda info: info["dev_bleu"])
print("best learning rate:", best_run["learning_rate"], "| dev BLEU:", round(best_run["dev_bleu"], 4))
model = AutoModelForSeq2SeqLM.from_pretrained(Path(best_run["run_dir"]) / "best").to(DEVICE).eval()

write_jsonl(t5_predict(model, dev), OUTPUT_DIR / f"{NAME}-dev.jsonl")
write_jsonl(t5_predict(model, test), OUTPUT_DIR / f"{NAME}-test.jsonl")
write_jsonl(t5_predict(model, test, use_gold_types=True), OUTPUT_DIR / f"{NAME}-goldtypes-test.jsonl")
(OUTPUT_DIR / f"{NAME}-settings.json").write_text(json.dumps(
    {"model": MODEL, "learning_rate": best_run["learning_rate"], "max_source_tokens": MAX_SOURCE,
     "max_target_tokens": MAX_TARGET, "num_beams": NUM_BEAMS}, indent=2) + "\n")
del model
free_gpu()
print(sorted(p.name for p in OUTPUT_DIR.glob(f"{NAME}*")))

best learning rate: 0.0003 | dev BLEU: 0.4505


Loading weights: 100%|██████████| 558/558 [00:00<00:00, 3807.38it/s]


['flan-t5-dev.jsonl', 'flan-t5-goldtypes-test.jsonl', 'flan-t5-settings.json', 'flan-t5-sweep.json', 'flan-t5-test.jsonl']


## Scores

`score.py` (the Milestone 2 evaluation script, copied here) scores each prediction file exactly as from the command line: BLEU (the main number), METEOR, BERTScore, exact match and token F1, overall and per gold type, plus type balanced accuracy. The JSON files go to `output/scores/`. The smoke test skips BERTScore and scores only the posts it predicted.

In [8]:
def gold_file(split, records):
    if not SMOKE:
        return DATA_DIR / f"{split}.jsonl"
    path = WORK_DIR / f"gold-{split}.jsonl"  # the smoke test predicts only a few posts
    write_jsonl(records, path)
    return path


def run_score(split, records, name):
    cmd = [sys.executable, "score.py", "--gold", str(gold_file(split, records)),
           "--pred", str(OUTPUT_DIR / f"{name}-{split}.jsonl"),
           "--output", str(OUTPUT_DIR / "scores" / f"{name}-{split}.json")]
    if SMOKE:
        cmd.append("--no-bertscore")
    print(f"$ python score.py --gold {cmd[3]} --pred {cmd[5]}")
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)


run_score("dev", dev, NAME)
run_score("test", test, NAME)
run_score("test", test, f"{NAME}-goldtypes")

$ python score.py --gold /home/dhanush/project/data/splits/dev.jsonl --pred output/flan-t5-dev.jsonl
Scored 794 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                    794       333       318       143
bleu              0.4505    0.5984    0.3113    0.4157
meteor            0.4035    0.2552    0.3508    0.5568
bertscore         0.9224    0.9435    0.9033    0.9157
exact-match       0.3161    0.5315    0.1604    0.1608
token-f1          0.5565    0.6602    0.4371    0.5803

spoiler type: balanced accuracy 0.7648
  phrase   precision 0.8283  recall 0.7387  f1 0.7810
  passage  precision 0.7342  recall 0.8774  f1 0.7994
  multi    precision 0.8291  recall 0.6783  f1 0.7462

BLEU (all): 0.4505

$ python score.py --gold /home/dhanush/project/data/splits/test.jsonl --pred output/flan-t5-test.jsonl
Scored 1000 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                   1000       423       403    

## A few test predictions

Six random test posts with their gold and predicted spoilers, to sanity-check the numbers above.

In [9]:
test_preds = {p["uuid"]: p for p in load_jsonl(OUTPUT_DIR / f"{NAME}-test.jsonl")}
sample = pd.DataFrame([{
    "post": post_text(r), "gold type": spoiler_type(r), "predicted type": test_preds[r["uuid"]]["spoilerType"],
    "gold spoiler": as_text(r["spoiler"]), "predicted spoiler": as_text(test_preds[r["uuid"]]["spoiler"]),
} for r in test])
sample.sample(min(6, len(sample)), random_state=0)

,post,gold type,predicted type,gold spoiler,predicted spoiler
993,Inside our three-month effort to attend Apple's iPhone 7 launch party,passage,passage,"The truth though is that large tech companies, especially in Silicon Valley, often use access to...",At this stage WWDC is a sell out. We have allocated seats for media but those too are taken. Sor...
859,Mum takes 'funny' photo of toddler - breaks down over chilling truth behind it,passage,passage,"""She was practising for a lockdown drill at her preschool and what you should do if you are stuc...","""She was practising for a lockdown drill at her preschool and what you should do if you are stuc..."
298,Duterte plans to ‘SHOOT and KILL’ these people; find out who they are,passage,phrase,business owners that he will kill them if they continue to practice endo or contractualization,"business owners who practice contractualization or ""endo"" (end of contract)"
553,LOOK: This Is Definitely The Biggest Spring 2014 Trend,passage,phrase,popping Spring colors and mixed prints,Spring colors and mixed prints
672,Lot of Pokémon Go players demanding refunds after latest update due to this reason,passage,passage,"game would no longer be supported in unlocked, rooted or jail-broken devices","A warning that the game would no longer be supported in unlocked, rooted or jail-broken devices ..."
971,"Former ""SNL"" star heads to ""House of Lies""",phrase,phrase,Fred Armisen,Fred Armisen
